In [7]:
import pandas as pd
import duckdb
from IPython.display import display

In [25]:
path = (r"\Users\DELL\Downloads\archive")
aisles = pd.read_csv(r"\Users\DELL\Downloads\archive\aisles.csv")
departments = pd.read_csv(r"\Users\DELL\Downloads\archive\departments.csv")
order_products = pd.read_csv(r"\Users\DELL\Downloads\archive\order_products__prior.csv")
orders = pd.read_csv(r"\Users\DELL\Downloads\archive\orders.csv")
products = pd.read_csv(r"\Users\DELL\Downloads\archive\products.csv")
order_products

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0
...,...,...,...,...
32434484,3421083,39678,6,1
32434485,3421083,11352,7,0
32434486,3421083,4600,8,0
32434487,3421083,24852,9,1


1. Join products and aisles to show product names and their aisle names.

In [8]:
conn = duckdb.connect()
conn.sql("""
    SELECT p.product_name, a.aisle
    from products p
    JOIN aisles a
    ON p.aisle_id = a.aisle_id
    """)

┌───────────────────────────────────────────────────────────────────┬───────────────────────────────┐
│                           product_name                            │             aisle             │
│                              varchar                              │            varchar            │
├───────────────────────────────────────────────────────────────────┼───────────────────────────────┤
│ Chocolate Sandwich Cookies                                        │ cookies cakes                 │
│ All-Seasons Salt                                                  │ spices seasonings             │
│ Robust Golden Unsweetened Oolong Tea                              │ tea                           │
│ Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce │ frozen meals                  │
│ Green Chile Anytime Sauce                                         │ marinades meat preparation    │
│ Dry Nose Oil                                                      │ cold flu all

2. Show the total number of products in each department.


In [17]:
dp = conn.sql("""
    SELECT p.product_name, d.department
    FROM products p
    JOIN departments d
    ON p.department_id = d.department_id
""")

In [22]:
conn.sql("""
    SELECT d.department, COUNT(d.product_name) 
    FROM dp d
    GROUP BY d.department
""")

┌─────────────────┬───────────────────────┐
│   department    │ count(d.product_name) │
│     varchar     │         int64         │
├─────────────────┼───────────────────────┤
│ meat seafood    │                   907 │
│ breakfast       │                  1115 │
│ canned goods    │                  2092 │
│ babies          │                  1081 │
│ personal care   │                  6563 │
│ pets            │                   972 │
│ missing         │                  1258 │
│ deli            │                  1322 │
│ bakery          │                  1516 │
│ household       │                  3085 │
│ other           │                   548 │
│ bulk            │                    38 │
│ frozen          │                  4007 │
│ dry goods pasta │                  1858 │
│ produce         │                  1684 │
│ international   │                  1139 │
│ snacks          │                  6264 │
│ pantry          │                  5371 │
│ alcohol         │             

3. List the names of products in the"alcohol department.


In [23]:
conn.sql("""
    SELECT d.product_name, d.department
    FROM dp d
    WHERE d.department = 'alcohol'
""")

┌───────────────────────────────────────────┬────────────┐
│               product_name                │ department │
│                  varchar                  │  varchar   │
├───────────────────────────────────────────┼────────────┤
│ Mirabelle Brut Rose                       │ alcohol    │
│ Chardonnay Paso Robles                    │ alcohol    │
│ Brut Rosé                                 │ alcohol    │
│ Tennessee Whiskey                         │ alcohol    │
│ Pinot Grigio, California, 2010            │ alcohol    │
│ Mixed 12 Pack Lion's Share Ale            │ alcohol    │
│ Super Dry Beer                            │ alcohol    │
│ Born Yesterday Pale Ale                   │ alcohol    │
│ Lite Beer                                 │ alcohol    │
│ Organic Reposado Tequila                  │ alcohol    │
│      ·                                    │    ·       │
│      ·                                    │    ·       │
│      ·                                    │    ·      

4. Find the average number of items per order


In [44]:
po = conn.sql("""
    SELECT p.product_name, o.order_id, o.add_to_cart_order
    FROM products p
    JOIN order_products o
    ON p.product_id = o.product_id
""")

In [53]:
nnum_product = conn.sql("""
    SELECT order_id, count(*) as num_products
    FROM po
    GROUP BY po.order_id
    """)

In [49]:
conn.sql("""
    SELECT avg(num_products) as avg_num_products
    FROM num_product
""")

┌────────────────────┐
│  avg_num_products  │
│       double       │
├────────────────────┤
│ 10.088883421247614 │
└────────────────────┘

5. List users who have placed more than 5 orders.


In [55]:
conn.sql("""
    SELECT order_id, num_products
    FROM num_product
    WHERE num_products > 5
""")

┌──────────┬──────────────┐
│ order_id │ num_products │
│  int64   │    int64     │
├──────────┼──────────────┤
│    75813 │            9 │
│    75835 │            6 │
│    75899 │            9 │
│    75900 │           14 │
│    75904 │           10 │
│    76061 │           14 │
│    76142 │           19 │
│    76233 │            8 │
│    76258 │           15 │
│    76288 │           14 │
│      ·   │            · │
│      ·   │            · │
│      ·   │            · │
│    36544 │            8 │
│    36610 │           18 │
│    36668 │           19 │
│    36669 │           16 │
│    36784 │           12 │
│    36816 │           19 │
│    36864 │           11 │
│    36896 │           20 │
│    36897 │            6 │
│    36915 │           10 │
└──────────┴──────────────┘
  ? rows        2 columns
  (>9999 rows, 20 shown)  

6. Join order_products and products to find the most frequently ordered product


In [58]:
op = conn.sql("""
    SELECT p.product_name, o.order_id
    FROM products p
    JOIN order_products o
    ON p.product_id = o.product_id
""")

In [60]:
conn.sql("""
    SELECT op.product_name, count(*) as num_orders
    FROM op
    GROUP BY op.product_name
    LIMIT 1
""")

┌──────────────────────┬────────────┐
│     product_name     │ num_orders │
│       varchar        │   int64    │
├──────────────────────┼────────────┤
│ Coldbrew Coffee Pops │       1255 │
└──────────────────────┴────────────┘

7. Find the average order_hour_of_day for each day of the week


In [86]:
conn.sql("""
      SELECT order_dow,AVG(order_hour_of_day) AS avg_order_hour
    FROM orders
    GROUP BY order_dow
    ORDER BY order_dow
    
""")

┌───────────┬────────────────────┐
│ order_dow │   avg_order_hour   │
│   int64   │       double       │
├───────────┼────────────────────┤
│         0 │ 13.573992561220159 │
│         1 │ 13.176062763201346 │
│         2 │ 13.467765269871164 │
│         3 │ 13.522555220929487 │
│         4 │  13.58528072730855 │
│         5 │ 13.360157752642445 │
│         6 │ 13.531044364372127 │
└───────────┴────────────────────┘

8. List all products have never been reordered



In [90]:
conn.sql("""
    SELECT DISTINCT op.product_id, p.product_name
    FROM order_products op
    JOIN products p
    ON op.product_id = p.product_id
    WHERE reordered = 0;
  """)

┌────────────┬───────────────────────────────────────────────────────────────────────────┐
│ product_id │                               product_name                                │
│   int64    │                                  varchar                                  │
├────────────┼───────────────────────────────────────────────────────────────────────────┤
│      34551 │ Organic Cane Sugar                                                        │
│        890 │ Organic Diced Tomatoes                                                    │
│      15057 │ Fresh Care Flushable Cleansing Cloths Refills                             │
│      18708 │ Organic Turmeric Ginger & Beet Superfoods Bar                             │
│      19701 │ Thin & Crispy Spinach & Garlic Pizza                                      │
│      32534 │ Nutz Over Chocolate Whole Nutrition Bars for Women                        │
│       5450 │ Small Hass Avocado                                                        │